# Phase 3 — Integrated Dataset Preparation

## Objective

Prepare the final modeling dataset by integrating:

- GraphTCDR multi-omics features for 28 matched pancreatic cancer cell lines
- PRISM drug-response data
- 2,048-bit Morgan fingerprints for 1,228 drugs

## Workflow

1. Load the four multi-omics feature matrices
2. Verify the 28 pancreatic cell lines and their feature dimensions
3. Load the processed drug fingerprints
4. Match drugs between PRISM and the molecular feature table
5. Prepare the log10(IC50) response target
6. Construct the cell–drug modeling dataset
7. Create and save a fixed cold-drug split

The resulting dataset will be used for baseline machine learning and GraphTCDR-based modeling.

## 1. Load Required Libraries

We first load the libraries needed for:

- Data handling with Pandas and NumPy
- Loading GraphTCDR `.pt` feature tensors with PyTorch
- Working with file paths using `pathlib`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)

print("\nEnvironment ready.")

NumPy: 2.2.6
Pandas: 2.3.3
PyTorch: 2.14.0+cpu

Environment ready.


### 2. Define Project Paths



- Raw PRISM data
- GraphTCDR multi-omics features
- Processed drug molecular features
- Output files for the integrated dataset

In [2]:
# Project directories
project_dir = Path("..")

raw_dir = project_dir / "data" / "raw"
processed_dir = project_dir / "data" / "processed"
results_dir = project_dir / "results"

# Check that the important directories exist
print("Raw data:", raw_dir.resolve())
print("Processed data:", processed_dir.resolve())
print("Results:", results_dir.resolve())

print("\nDirectories ready.")

Raw data: D:\Pancreatic_GraphTCDR\data\raw
Processed data: D:\Pancreatic_GraphTCDR\data\processed
Results: D:\Pancreatic_GraphTCDR\results

Directories ready.


### 3. Load GraphTCDR Multi-Omics Data

GraphTCDR provides processed feature matrices for 402 cancer cell lines:

- Gene expression
- Copy number variation (CNV)
- miRNA
- DNA methylation

We will load these matrices and later select the **28 pancreatic cancer cell lines** that were successfully matched with our PRISM data.

In [3]:
# Load GraphTCDR cell-line mapping
graphtcdr_cells = pd.read_csv(
    raw_dir / "cellLineName402.csv"
)

# Load the four multi-omics feature matrices
gene_expr = torch.load(
    raw_dir / "GeneExpressionFeature402.pt",
    weights_only=True
)

cnv = torch.load(
    raw_dir / "CNVFeature402.pt",
    weights_only=True
)

mirna = torch.load(
    raw_dir / "miRNAFeature402.pt",
    weights_only=True
)

methylation = torch.load(
    raw_dir / "MethylationFeature402.pt",
    weights_only=True
)

print("Cell-line mapping:", graphtcdr_cells.shape)
print("Gene expression:", gene_expr.shape)
print("CNV:", cnv.shape)
print("miRNA:", mirna.shape)
print("Methylation:", methylation.shape)

Cell-line mapping: (402, 2)
Gene expression: torch.Size([402, 20404])
CNV: torch.Size([402, 22582])
miRNA: torch.Size([402, 734])
Methylation: torch.Size([402, 20587])


### 4. Identify the 28 Matched Pancreatic Cell Lines

The GraphTCDR dataset contains 402 cell lines, while our PRISM pancreatic subset contains 37 pancreatic cell lines.

Only **28 cell lines are present in both datasets**.

We will use these 28 matched cell lines so that the PRISM response data and GraphTCDR multi-omics features refer to the same cell lines.

In [4]:
# Get pancreatic cell lines from the PRISM cell-line metadata
prism_cells = pd.read_csv(
    raw_dir / "secondary-screen-cell-line-info.csv"
)

pancreatic_cells = prism_cells[
    prism_cells["primary_tissue"].astype(str).str.upper() == "PANCREAS"
].copy()

# Match GraphTCDR cell lines with PRISM pancreatic cell lines
pancreatic_names = set(
    pancreatic_cells["ccle_name"]
    .astype(str)
    .str.upper()
)

graphtcdr_pancreatic = graphtcdr_cells[
    graphtcdr_cells["cellLineName"]
    .astype(str)
    .str.upper()
    .isin(pancreatic_names)
].copy()

# Preserve the original GraphTCDR row indices
pancreatic_indices = graphtcdr_pancreatic["index"].to_numpy()

print("PRISM pancreatic cell lines:", len(pancreatic_cells))
print("Matched GraphTCDR pancreatic cell lines:", len(graphtcdr_pancreatic))

print("\nMatched cell lines:")
print(graphtcdr_pancreatic["cellLineName"].to_list())

PRISM pancreatic cell lines: 37
Matched GraphTCDR pancreatic cell lines: 28

Matched cell lines:
['PATU8988S_PANCREAS', 'PANC0327_PANCREAS', 'PANC1_PANCREAS', 'PSN1_PANCREAS', 'SUIT2_PANCREAS', 'SNU213_PANCREAS', 'PANC0813_PANCREAS', 'DANG_PANCREAS', 'HUPT3_PANCREAS', 'CAPAN2_PANCREAS', 'PK45H_PANCREAS', 'PANC0403_PANCREAS', 'L33_PANCREAS', 'SNU410_PANCREAS', 'TCCPAN2_PANCREAS', 'BXPC3_PANCREAS', 'PATU8902_PANCREAS', 'QGP1_PANCREAS', 'CFPAC1_PANCREAS', 'PANC1005_PANCREAS', 'HPAC_PANCREAS', 'SW1990_PANCREAS', 'PK1_PANCREAS', 'PATU8988T_PANCREAS', 'ASPC1_PANCREAS', 'YAPC_PANCREAS', 'PANC0203_PANCREAS', 'HS766T_PANCREAS']


### 5. Extract Multi-Omics Features for the 28 Matched Cell Lines

The original GraphTCDR tensors contain 402 cell lines.

Using the verified GraphTCDR row indices, we now extract only the 28 pancreatic cell lines that are also present in PRISM.

The resulting matrices will contain:

- Gene expression: 28 × 20,404
- CNV: 28 × 22,582
- miRNA: 28 × 734
- DNA methylation: 28 × 20,587

In [5]:
# Extract the 28 matched pancreatic cell lines
pancreatic_gene_expr = gene_expr[pancreatic_indices]
pancreatic_cnv = cnv[pancreatic_indices]
pancreatic_mirna = mirna[pancreatic_indices]
pancreatic_methylation = methylation[pancreatic_indices]

print("Pancreatic gene expression:", pancreatic_gene_expr.shape)
print("Pancreatic CNV:", pancreatic_cnv.shape)
print("Pancreatic miRNA:", pancreatic_mirna.shape)
print("Pancreatic methylation:", pancreatic_methylation.shape)

Pancreatic gene expression: torch.Size([28, 20404])
Pancreatic CNV: torch.Size([28, 22582])
Pancreatic miRNA: torch.Size([28, 734])
Pancreatic methylation: torch.Size([28, 20587])


### 6. Verify Cell-Line Ordering

The GraphTCDR feature tensors use the row order defined by `cellLineName402.csv`.

We verify that the stored indices are sequential and therefore correctly map each extracted feature row to its corresponding cell line.

In [6]:
# Verify GraphTCDR row ordering
index_check = (
    graphtcdr_cells["index"].to_numpy() == np.arange(len(graphtcdr_cells))
).all()

print("GraphTCDR row ordering verified:", index_check)

# Display the mapping used for the pancreatic subset
cell_mapping = graphtcdr_pancreatic[
    ["index", "cellLineName"]
].reset_index(drop=True)

print("\nPancreatic cell-line mapping:")
display(cell_mapping)

GraphTCDR row ordering verified: True

Pancreatic cell-line mapping:


,index,cellLineName
0,0,PATU8988S_PANCREAS
1,39,PANC0327_PANCREAS
2,47,PANC1_PANCREAS
3,68,PSN1_PANCREAS
4,79,SUIT2_PANCREAS
5,84,SNU213_PANCREAS
6,85,PANC0813_PANCREAS
7,91,DANG_PANCREAS
8,103,HUPT3_PANCREAS
9,123,CAPAN2_PANCREAS


### 7. Save the 28-Cell-Line Multi-Omics Features

The extracted multi-omics matrices are now ready.

We save them separately so they can be reused during model development without repeatedly loading the full 402-cell-line GraphTCDR tensors.

In [7]:
# Save the pancreatic multi-omics features
torch.save(
    pancreatic_gene_expr,
    processed_dir / "pancreatic28_gene_expression.pt"
)

torch.save(
    pancreatic_cnv,
    processed_dir / "pancreatic28_cnv.pt"
)

torch.save(
    pancreatic_mirna,
    processed_dir / "pancreatic28_mirna.pt"
)

torch.save(
    pancreatic_methylation,
    processed_dir / "pancreatic28_methylation.pt"
)

cell_mapping.to_csv(
    processed_dir / "pancreatic28_cell_mapping.csv",
    index=False
)

print("Saved 28-cell-line multi-omics features successfully.")

Saved 28-cell-line multi-omics features successfully.


### 8. Load Processed Drug Molecular Features

Phase 2 produced molecular features for 1,228 PRISM drugs.

Each drug is represented by a **2,048-bit Morgan fingerprint** generated from its canonical SMILES.

We now load these processed features so they can be linked with the PRISM drug-response records.

In [8]:
# Load processed drug molecular features
drug_fingerprints = np.load(
    processed_dir / "drug_morgan_fingerprints_1228x2048.npy"
)

drug_smiles = pd.read_csv(
    processed_dir / "drug_smiles_1228.csv"
)

print("Drug fingerprint matrix:", drug_fingerprints.shape)
print("Drug SMILES table:", drug_smiles.shape)

print("\nFingerprint dtype:", drug_fingerprints.dtype)
print("Number of drugs:", len(drug_smiles))

Drug fingerprint matrix: (1228, 2048)
Drug SMILES table: (1228, 2)

Fingerprint dtype: uint8
Number of drugs: 1228


### 9. Load PRISM Drug-Response Data

We now load the PRISM dose-response data and keep only the 28 pancreatic cell lines that have corresponding GraphTCDR multi-omics features.

For modeling, we will use finite IC50 measurements and retain the original PRISM drug names so they can be matched with the molecular fingerprint table.

In [9]:
# Load PRISM drug-response data
response_path = raw_dir / "secondary-screen-dose-response-curve-parameters.csv"

prism_response = pd.read_csv(response_path)

print("Full PRISM response shape:", prism_response.shape)

# Keep only the 28 matched GraphTCDR pancreatic cell lines
matched_depmap_ids = pancreatic_cells[
    pancreatic_cells["ccle_name"].isin(
        graphtcdr_pancreatic["cellLineName"]
    )
]["depmap_id"]

prism_28 = prism_response[
    prism_response["depmap_id"].isin(matched_depmap_ids)
].copy()

print("28-cell-line PRISM response shape:", prism_28.shape)
print("Unique cell lines:", prism_28["depmap_id"].nunique())
print("Unique drugs:", prism_28["name"].nunique())

Full PRISM response shape: (701004, 20)
28-cell-line PRISM response shape: (40119, 20)
Unique cell lines: 28
Unique drugs: 1448


C:\Users\SREEKUTTY\AppData\Local\Temp\ipykernel_10984\3577896065.py:4: DtypeWarning: Columns (14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  prism_response = pd.read_csv(response_path)


### 10. Prepare the IC50 Response Target

IC50 is the primary response variable for this project.

We first convert the IC50 column to numeric values and remove non-finite values such as missing values and infinity.

The modeling target will be:

**log10(IC50)**

The logarithmic transformation reduces the numerical scale of IC50 and allows us to examine the highly skewed response distribution more effectively.

In [10]:
# Convert IC50 to numeric
prism_28["ic50"] = pd.to_numeric(
    prism_28["ic50"],
    errors="coerce"
)

# Keep only finite IC50 values
usable_response = prism_28[
    np.isfinite(prism_28["ic50"])
].copy()

# Create log10(IC50) target
usable_response["log_ic50"] = np.log10(
    usable_response["ic50"]
)

print("Total 28-cell-line records:", len(prism_28))
print("Finite IC50 records:", len(usable_response))
print("Unique cell lines:", usable_response["depmap_id"].nunique())
print("Unique drugs:", usable_response["name"].nunique())

print("\nlog10(IC50) summary:")
print(usable_response["log_ic50"].describe())

Total 28-cell-line records: 40119
Finite IC50 records: 18912
Unique cell lines: 28
Unique drugs: 1228

log10(IC50) summary:
count    18912.000000
mean         0.005375
std          3.085030
min       -110.746563
25%         -0.353090
50%          0.197369
75%          0.511461
max        287.762339
Name: log_ic50, dtype: float64


## 11. Inspect Response Quality Across the IC50 Distribution

Extreme IC50 values may correspond to poorly fitted dose-response curves.

Therefore, before deciding how to handle the tail, we examine the relationship between **log10(IC50)** and the PRISM **R²** curve-fit quality.

This helps distinguish the response distribution from potential low-quality fitted measurements.

In [11]:
# Create bins across the log10(IC50) distribution
bins = [-np.inf, -3, -2, -1, 0, 1, 2, 3, np.inf]
labels = [
    "< -3",
    "-3 to -2",
    "-2 to -1",
    "-1 to 0",
    "0 to 1",
    "1 to 2",
    "2 to 3",
    "> 3"
]

usable_response["log_ic50_bin"] = pd.cut(
    usable_response["log_ic50"],
    bins=bins,
    labels=labels
)

# Convert R² to numeric
usable_response["r2"] = pd.to_numeric(
    usable_response["r2"],
    errors="coerce"
)

# Summarize response quality by IC50 region
tail_summary = (
    usable_response
    .groupby("log_ic50_bin", observed=False)["r2"]
    .agg(
        count="count",
        median_r2="median",
        mean_r2="mean"
    )
    .reset_index()
)

print(tail_summary)

  log_ic50_bin  count  median_r2   mean_r2
0         < -3    225   0.108885  0.226843
1     -3 to -2    428   0.766693  0.693731
2     -2 to -1   1615   0.738775  0.690050
3      -1 to 0   5240   0.608689  0.576298
4       0 to 1  11082   0.411352  0.379779
5       1 to 2    216   0.123736  0.156980
6       2 to 3     34   0.001938 -0.421490
7          > 3     72  -0.012675 -0.041088


### 12. Create the Cell–Drug Response Table

We now reduce the PRISM records to the variables required for integration:

- `depmap_id` — cell line
- `name` — drug
- `smiles` — drug structure
- `ic50` — original IC50
- `log_ic50` — modeling target
- `r2` — dose-response curve-fit quality

This table will become the response component of the integrated dataset.

In [12]:
# Select variables needed for integration
response_table = usable_response[
    [
        "depmap_id",
        "ccle_name",
        "name",
        "smiles",
        "ic50",
        "log_ic50",
        "r2"
    ]
].copy()

# Check for duplicate cell-drug combinations
duplicate_pairs = response_table.duplicated(
    subset=["depmap_id", "name"]
).sum()

print("Response records:", len(response_table))
print("Unique cell-drug pairs:",
      response_table[["depmap_id", "name"]].drop_duplicates().shape[0])
print("Duplicate cell-drug records:", duplicate_pairs)

Response records: 18912
Unique cell-drug pairs: 18037
Duplicate cell-drug records: 875


### 13. Resolve Duplicate Cell–Drug Measurements

Some cell–drug combinations have more than one PRISM record.

Before building the final response matrix, we will inspect how many records occur per cell–drug pair. This allows us to determine whether duplicates are isolated cases or whether some pairs have many repeated measurements.

In [14]:
# Count records for each cell-drug pair
pair_counts = (
    response_table
    .groupby(["depmap_id", "name"])
    .size()
    .reset_index(name="n_records")
)

print("Cell-drug pairs:", len(pair_counts))
print("Pairs with duplicates:", (pair_counts["n_records"] > 1).sum())
print("Maximum records for one pair:", pair_counts["n_records"].max())

print("\nDistribution of records per cell-drug pair:")
print(pair_counts["n_records"].value_counts().sort_index())

Cell-drug pairs: 18037
Pairs with duplicates: 872
Maximum records for one pair: 3

Distribution of records per cell-drug pair:
n_records
1    17165
2      869
3        3
Name: count, dtype: int64


### 14. Inspect Duplicate Cell–Drug Measurements

We inspect the duplicated cell–drug pairs to determine whether their IC50 values differ.

This is important because averaging repeated measurements is only appropriate if the records represent comparable measurements.

In [15]:
# Get duplicated cell-drug pairs
duplicate_pair_keys = pair_counts[
    pair_counts["n_records"] > 1
][["depmap_id", "name"]]

duplicate_records = response_table.merge(
    duplicate_pair_keys,
    on=["depmap_id", "name"],
    how="inner"
)

# Display the first duplicated pairs
duplicate_records[
    [
        "depmap_id",
        "ccle_name",
        "name",
        "ic50",
        "log_ic50",
        "r2"
    ]
].sort_values(
    ["depmap_id", "name"]
).head(20)

,depmap_id,ccle_name,name,ic50,log_ic50,r2
787,ACH-000022,PATU8988S_PANCREAS,AS-703026,0.120511,-0.918974,0.835624
1057,ACH-000022,PATU8988S_PANCREAS,AS-703026,0.547658,-0.261490,0.532619
316,ACH-000022,PATU8988S_PANCREAS,AT-7519,2.467227,0.392209,0.648542
1038,ACH-000022,PATU8988S_PANCREAS,AT-7519,0.629852,-0.200762,0.586774
435,ACH-000022,PATU8988S_PANCREAS,AT13387,0.144085,-0.841382,0.534997
1047,ACH-000022,PATU8988S_PANCREAS,AT13387,0.341300,-0.466863,0.610585
454,ACH-000022,PATU8988S_PANCREAS,FK-866,0.032338,-1.490290,0.753232
1048,ACH-000022,PATU8988S_PANCREAS,FK-866,0.071178,-1.147651,0.457490
694,ACH-000022,PATU8988S_PANCREAS,LY2603618,3.156491,0.499205,-0.118124
1040,ACH-000022,PATU8988S_PANCREAS,LY2603618,0.720806,-0.142182,0.643920


### 15. Aggregate Repeated Cell–Drug Measurements

Some cell–drug combinations have multiple IC50 measurements.

To obtain one response value for each cell–drug pair, we use the **median of log10(IC50)** across repeated measurements.

Using the median reduces the influence of unusually high or low repeated measurements while retaining the available response information.

In [16]:
# Aggregate repeated measurements using the median log10(IC50)
response_table_agg = (
    response_table
    .groupby(
        ["depmap_id", "ccle_name", "name"],
        as_index=False
    )
    .agg(
        log_ic50=("log_ic50", "median"),
        r2=("r2", "median")
    )
)

print("Original response records:", len(response_table))
print("Unique cell-drug pairs:", len(response_table_agg))

print("\nAggregated response table:")
display(response_table_agg.head())

Original response records: 18912
Unique cell-drug pairs: 18037

Aggregated response table:


,depmap_id,ccle_name,name,log_ic50,r2
0,ACH-000022,PATU8988S_PANCREAS,1-naphthyl-PP1,0.568421,-0.255038
1,ACH-000022,PATU8988S_PANCREAS,1-phenylbiguanide,1.092774,-0.139651
2,ACH-000022,PATU8988S_PANCREAS,10-deacetylbaccatin,0.158021,0.385270
3,ACH-000022,PATU8988S_PANCREAS,10-hydroxycamptothecin,-1.413141,0.852317
4,ACH-000022,PATU8988S_PANCREAS,5-hydroxymethyl-tolterodine,-0.201739,-0.500489


### 16. Match PRISM Drugs with Molecular Fingerprints

The response table contains 1,228 unique drugs, while the processed molecular feature table also contains 1,228 drugs.

We now verify that every drug appearing in the response data has a corresponding 2,048-bit Morgan fingerprint before constructing the integrated dataset.

In [17]:
# Check whether all response drugs are present in the fingerprint table
response_drugs = set(response_table_agg["name"])
fingerprint_drugs = set(drug_smiles["name"])

missing_drugs = response_drugs - fingerprint_drugs
extra_drugs = fingerprint_drugs - response_drugs

print("Unique response drugs:", len(response_drugs))
print("Drugs with fingerprints:", len(fingerprint_drugs))
print("Missing fingerprints:", len(missing_drugs))
print("Fingerprint-only drugs:", len(extra_drugs))

if len(missing_drugs) == 0:
    print("\nAll response drugs have molecular fingerprints.")
else:
    print("\nMissing drugs:")
    print(sorted(missing_drugs))

Unique response drugs: 1228
Drugs with fingerprints: 1228
Missing fingerprints: 0
Fingerprint-only drugs: 0

All response drugs have molecular fingerprints.


### 17. Create the Final Integrated Cell–Drug Dataset

We now combine the PRISM response table with the 2,048-bit molecular fingerprint representation of each drug.

Each row will represent one unique **cell line–drug pair** with:

- Cell line identifier
- Drug name
- log10(IC50) response
- Curve-fit R²
- 2,048 molecular fingerprint features

The multi-omics cell-line features will be linked separately using the verified 28-cell-line mapping.

In [18]:
# Create a lookup table from drug name to fingerprint
drug_fp_lookup = {
    drug_name: drug_fingerprints[i]
    for i, drug_name in enumerate(drug_smiles["name"])
}

# Attach the molecular fingerprint to every cell-drug response pair
response_table_agg["drug_fp"] = response_table_agg["name"].map(
    drug_fp_lookup
)

# Verify that every response pair received a fingerprint
missing_fp_rows = response_table_agg["drug_fp"].isna().sum()

print("Integrated response rows:", len(response_table_agg))
print("Rows missing drug fingerprints:", missing_fp_rows)

# Check one example
print("\nExample fingerprint length:",
      len(response_table_agg["drug_fp"].iloc[0]))

Integrated response rows: 18037
Rows missing drug fingerprints: 0

Example fingerprint length: 2048


### 18. Link Multi-Omics Features to Each Cell–Drug Pair

Each response row belongs to one of the 28 matched pancreatic cell lines.

We use the verified GraphTCDR cell-line mapping to connect every response row with its corresponding:

- Gene expression profile
- CNV profile
- miRNA profile
- DNA methylation profile

The same cell-line feature vector is therefore reused for all drugs tested on that cell line.

In [19]:
# Create a mapping from DepMap ID to the corresponding GraphTCDR row
depmap_to_omics_index = dict(
    zip(
        pancreatic_cells["depmap_id"],
        pancreatic_cells["ccle_name"]
    )
)

# Check that all response cell lines belong to our 28 matched cell lines
response_cell_lines = set(response_table_agg["depmap_id"])
matched_cell_lines = set(matched_depmap_ids)

missing_cell_lines = response_cell_lines - matched_cell_lines

print("Response cell lines:", len(response_cell_lines))
print("Matched cell lines:", len(matched_cell_lines))
print("Response cell lines without omics:", len(missing_cell_lines))

if len(missing_cell_lines) == 0:
    print("\nAll response cell lines have multi-omics features.")

Response cell lines: 28
Matched cell lines: 28
Response cell lines without omics: 0

All response cell lines have multi-omics features.


### 19. Create Cell-Line Multi-Omics Lookup

We now create a direct lookup from each PRISM `depmap_id` to its four corresponding GraphTCDR feature vectors.

This allows us to retrieve the correct multi-omics representation whenever a cell–drug response pair is processed.

In [21]:
# Create a clean mapping using only the 28 matched cell lines
matched_cell_mapping = graphtcdr_pancreatic[
    ["index", "cellLineName"]
].copy()

# Match GraphTCDR cell names to PRISM DepMap IDs
matched_cell_mapping = matched_cell_mapping.merge(
    pancreatic_cells[["depmap_id", "ccle_name"]],
    left_on="cellLineName",
    right_on="ccle_name",
    how="left"
)

# Create DepMap ID -> GraphTCDR index mapping
depmap_to_index = dict(
    zip(
        matched_cell_mapping["depmap_id"],
        matched_cell_mapping["index"].astype(int)
    )
)

print("Number of DepMap-to-omics mappings:", len(depmap_to_index))

print("\nExample mappings:")
for depmap_id, graph_index in list(depmap_to_index.items())[:5]:
    print(depmap_id, "-> GraphTCDR index", graph_index)

Number of DepMap-to-omics mappings: 28

Example mappings:
ACH-000022 -> GraphTCDR index 0
ACH-000139 -> GraphTCDR index 39
ACH-000164 -> GraphTCDR index 47
ACH-000320 -> GraphTCDR index 68
ACH-000652 -> GraphTCDR index 79


## 20. Verify the Multi-Omics Mapping

Before attaching the high-dimensional feature vectors to the response table, we verify that every response row maps to a valid GraphTCDR feature row.

This is a final consistency check before constructing the integrated dataset.

In [22]:
# Map every response row to its GraphTCDR feature index
response_table_agg["omics_index"] = response_table_agg["depmap_id"].map(
    depmap_to_index
)

# Verify the mapping
missing_omics = response_table_agg["omics_index"].isna().sum()

print("Total response rows:", len(response_table_agg))
print("Rows without omics mapping:", missing_omics)

print(
    "Unique GraphTCDR indices used:",
    response_table_agg["omics_index"].nunique()
)

print("\nMapping verification:")
if missing_omics == 0 and response_table_agg["omics_index"].nunique() == 28:
    print("All 18,037 response pairs are correctly mapped to 28 GraphTCDR cell-line feature rows.")
else:
    print("Mapping requires further inspection.")

Total response rows: 18037
Rows without omics mapping: 0
Unique GraphTCDR indices used: 28

Mapping verification:
All 18,037 response pairs are correctly mapped to 28 GraphTCDR cell-line feature rows.


### 21. Prepare the Final Modeling Response Table

We now create a compact table containing the identifiers and targets needed for modeling.

The high-dimensional omics and molecular fingerprint arrays will remain stored separately rather than being expanded into thousands of dataframe columns.

This keeps the integrated table manageable and allows the modeling pipeline to retrieve the corresponding feature vectors using the cell-line and drug indices.

In [23]:
# Create compact identifiers for drugs
drug_to_index = {
    drug_name: i
    for i, drug_name in enumerate(drug_smiles["name"])
}

# Add drug fingerprint index
response_table_agg["drug_index"] = response_table_agg["name"].map(
    drug_to_index
)

# Create the final compact modeling table
modeling_table = response_table_agg[
    [
        "depmap_id",
        "ccle_name",
        "omics_index",
        "name",
        "drug_index",
        "log_ic50",
        "r2"
    ]
].copy()

print("Final modeling table shape:", modeling_table.shape)
print("\nColumns:")
print(modeling_table.columns.tolist())

print("\nMissing values:")
print(modeling_table.isna().sum())

Final modeling table shape: (18037, 7)

Columns:
['depmap_id', 'ccle_name', 'omics_index', 'name', 'drug_index', 'log_ic50', 'r2']

Missing values:
depmap_id      0
ccle_name      0
omics_index    0
name           0
drug_index     0
log_ic50       0
r2             0
dtype: int64


### 22. Create a Fixed Cold-Drug Split

The main evaluation strategy is a **cold-drug split**.

All response records belonging to the same drug will be assigned to the same partition. Therefore, drugs in the test set are completely unseen during training.

This is important because the project evaluates how well the model generalizes to previously unseen drugs.

The split will be created once, saved to disk, and reused for all models so that different approaches are compared on exactly the same test drugs.

In [24]:
from sklearn.model_selection import GroupShuffleSplit

# Fixed random seed for reproducibility
RANDOM_STATE = 42

# Create a 80/20 cold-drug split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(
        modeling_table,
        groups=modeling_table["name"]
    )
)

# Create train and test tables
train_table = modeling_table.iloc[train_idx].reset_index(drop=True)
test_table = modeling_table.iloc[test_idx].reset_index(drop=True)

# Verify that no drug appears in both sets
train_drugs = set(train_table["name"])
test_drugs = set(test_table["name"])

overlap = train_drugs.intersection(test_drugs)

print("Total drugs:", modeling_table["name"].nunique())
print("Training drugs:", len(train_drugs))
print("Test drugs:", len(test_drugs))
print("Drug overlap:", len(overlap))

print("\nTraining response pairs:", len(train_table))
print("Test response pairs:", len(test_table))

Total drugs: 1228
Training drugs: 982
Test drugs: 246
Drug overlap: 0

Training response pairs: 14235
Test response pairs: 3802


### 23. Save the Fixed Cold-Drug Split

The drug-level split is created only once using a fixed random seed.

We save the train/test assignment so that every subsequent model uses exactly the same cold-drug split.

This prevents differences in model performance from being caused by different train/test partitions.

In [25]:
# Save the fixed split assignments
split_dir = processed_dir / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

train_table.to_csv(
    split_dir / "cold_drug_train.csv",
    index=False
)

test_table.to_csv(
    split_dir / "cold_drug_test.csv",
    index=False
)

print("Cold-drug split saved successfully.")

print("\nTrain file:")
print(split_dir / "cold_drug_train.csv")

print("\nTest file:")
print(split_dir / "cold_drug_test.csv")

Cold-drug split saved successfully.

Train file:
..\data\processed\splits\cold_drug_train.csv

Test file:
..\data\processed\splits\cold_drug_test.csv


### 24. Save the Integrated Modeling Table

We save the complete compact dataset before starting model development.

This file contains the response target and the indices needed to retrieve the corresponding cell-line multi-omics features and drug molecular fingerprints.

The high-dimensional feature matrices remain stored separately.

In [26]:
# Save the complete compact modeling table
modeling_table.to_csv(
    processed_dir / "integrated_modeling_table.csv",
    index=False
)

print("Integrated modeling table saved successfully.")
print(
    "File:",
    processed_dir / "integrated_modeling_table.csv"
)

Integrated modeling table saved successfully.
File: ..\data\processed\integrated_modeling_table.csv
